# Finite-Geometry AA-DPD PET Slab for PETase Interface Models

This notebook builds a role-aware all-atom PET dense periodic cell with MuPT, using an 8 x 8 x 2 nm box at the target PET density as the initialization geometry. The PBC vectors are only an initialization aid: the exported whole-chain CIF provides sensible PET slab coordinates for later protein/water setup workflows.

The target application is PETase at a water-bottle PET interface. The slab density target is 1.38 g/cm^3, and the lateral dimensions are 8 x 8 nm so a roughly 6 x 6 x 6 nm enzyme can be visually checked against a surface that is wider than the enzyme footprint.


## 1. Environment and Science Knobs

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import math
import sys
import time

import networkx as nx
import numpy as np
from anytree import PreOrderIter
from rdkit import Chem
from rdkit.Geometry import Point3D
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation

def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")

EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

from mupt.geometry.shapes import PointCloud
from mupt.interfaces.rdkit import primitive_to_rdkit_mols
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.muptio.sdf import prepare_mupt_sdf_atom_props
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole

try:
    import gsd.hoomd
    import hoomd
    HOOMD_AVAILABLE = True
except ModuleNotFoundError as exc:
    HOOMD_AVAILABLE = False
    HOOMD_IMPORT_ERROR = exc

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "finite_geometry_pet_slab_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 51
TARGET_DENSITY_G_CM3 = 1.38
SLAB_X_NM = 8.0
SLAB_Y_NM = 8.0
SLAB_Z_NM = 2.0
DPD_BOX_Z_NM = SLAB_Z_NM
CHAIN_REPEAT_UNITS = 12
RUN_AA_DPD = True
RUN_OPENMM_MINIMIZATION = True

DPD_STEPS_MAX = 50_000
DPD_STEPS_PER_INTERVAL = 1_000
DPD_DT = 0.001
DPD_R_CUT_A = 3.0
DPD_A = 5_000.0
DPD_GAMMA = 800.0
DPD_KT = 1.0
DPD_BOND_K = 500.0
DPD_BOND_K_SCALE = 1.0
DPD_ANGLE_K_SCALE = 1.0
DPD_DIHEDRAL_K_SCALE = 1.0
DPD_IMPROPER_K_SCALE = 1.0
MIN_INTERMOLECULAR_DISTANCE_A = 1.05
INITIAL_REPEAT_SPACING_A = 1.45

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"
OPENMM_PLATFORM_NAME = "CUDA"
OPENMM_PLATFORM_PROPERTIES = {"Precision": "mixed", "DeviceIndex": "0"}
OPENMM_MINIMIZATION_TOLERANCE_KJ_MOL_NM = 1.0e-3
OPENMM_MD_TEMPERATURE_K = 300.0
OPENMM_MD_FRICTION_PER_PS = 1.0
OPENMM_MD_TIMESTEP_FS = 2.0
OPENMM_MD_DURATION_NS = 0.5
OPENMM_MD_REPORT_INTERVAL_STEPS = 10_000
OPENMM_NPT_DURATION_NS = 0.5
OPENMM_NPT_PRESSURE_ATM = 1.0
OPENMM_BAROSTAT_FREQUENCY_STEPS = 25
TRAJECTORY_FRAMES = 150

DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076
slab_volume_nm3 = SLAB_X_NM * SLAB_Y_NM * SLAB_Z_NM
target_mass_da = TARGET_DENSITY_G_CM3 * slab_volume_nm3 / DA_PER_NM3_TO_G_CM3
pet_repeat_mass_da = 192.168  # C10H8O4 repeat mass, used only for chain-count sizing
N_CHAINS = max(1, round(target_mass_da / (CHAIN_REPEAT_UNITS * pet_repeat_mass_da)))
OPENMM_MD_STEPS = int(round(OPENMM_MD_DURATION_NS * 1_000_000.0 / OPENMM_MD_TIMESTEP_FS))
OPENMM_NPT_STEPS = int(round(OPENMM_NPT_DURATION_NS * 1_000_000.0 / OPENMM_MD_TIMESTEP_FS))

print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"HOOMD available: {HOOMD_AVAILABLE}")
if not HOOMD_AVAILABLE:
    print(f"  {HOOMD_IMPORT_ERROR}")
print(f"Target slab: {SLAB_X_NM} x {SLAB_Y_NM} x {SLAB_Z_NM} nm^3")
print(f"Target PET mass: {target_mass_da:.1f} Da; using {N_CHAINS} chains x {CHAIN_REPEAT_UNITS} repeat units")

## 2. Build Role-Aware PET Chains in MuPT

PET is represented as hydroxyl/carboxyl terminated oligomers with connector-marked head, middle, and tail residues. The `*` atoms mark polymerization linkers; atom-map labels give MuPT a consistent traversal direction for topology registration.

In [ ]:
PET_RESIDUE_SMILES = {
    "head": "[H]O[CH2][CH2]OC(=O)c1ccc([C:2](=O)-*)cc1",
    "mid": "*-[O:1][CH2][CH2]OC(=O)c1ccc([C:2](=O)-*)cc1",
    "tail": "*-[O:1][CH2][CH2]OC(=O)c1ccc([C:2](=O)O[H])cc1",
}
RESNAME_MAP = {"head": "PTH", "mid": "PET", "tail": "PTT"}

@dataclass(frozen=True)
class BuiltPETSlab:
    primitive: Primitive
    chain_sequences: list[list[str]]
    total_mass_da: float

def build_pet_lexicon() -> dict[str, Primitive]:
    lexicon = {}
    for name, smiles in PET_RESIDUE_SMILES.items():
        residue = primitive_from_smiles(smiles, ensure_explicit_Hs=True, embed_positions=True, label=name)
        residue.role = PrimitiveRole.RESIDUE
        residue.metadata["residue_name"] = RESNAME_MAP[name]
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
        lexicon[name] = residue
    return lexicon

def primitive_mass_da(root: Primitive) -> float:
    mass = 0.0
    for atom in root.leaves:
        if atom.element is None:
            raise ValueError(f"Atomic primitive {atom.label!r} has no element")
        mass += float(atom.element.mass)
    return mass

def build_pet_slab(n_chains: int, repeat_units: int) -> BuiltPETSlab:
    if repeat_units < 2:
        raise ValueError("repeat_units must be at least 2 for head/tail PET chains")
    lexicon = build_pet_lexicon()
    universe = Primitive(label="pet_water_bottle_interface_slab", role=PrimitiveRole.UNIVERSE)
    universe.metadata.update({
        "system_name": "PET_slab_8x8x2nm",
        "target_density_g_cm3": str(TARGET_DENSITY_G_CM3),
        "slab_dimensions_nm": json.dumps([SLAB_X_NM, SLAB_Y_NM, SLAB_Z_NM]),
        "dpd_box_dimensions_nm": json.dumps([SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM]),
        "placement_method": "FiniteGeometryAllAtomDPDPlacement",
    })
    chain_sequences = []
    for chain_idx in range(n_chains):
        segment = Primitive(label=f"pet_chain_{chain_idx:04d}", role=PrimitiveRole.SEGMENT)
        sequence = ["head", *(["mid"] * (repeat_units - 2)), "tail"]
        chain_sequences.append(sequence)
        handles = []
        for repeat_idx, residue_name in enumerate(sequence):
            residue = lexicon[residue_name].copy()
            residue.role = PrimitiveRole.RESIDUE
            residue.label = f"{residue_name}_{repeat_idx:03d}"
            residue.metadata.update({
                "residue_name": RESNAME_MAP[residue_name],
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
            })
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
            handles.append(segment.attach_child(residue))
        segment.set_topology(nx.path_graph(handles, create_using=TopologicalStructure), max_registration_iter=100)
        universe.attach_child(segment)
    total_mass = primitive_mass_da(universe)
    universe.metadata["total_mass_da"] = str(total_mass)
    universe.metadata["actual_initial_density_g_cm3"] = str(total_mass * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3)
    return BuiltPETSlab(primitive=universe, chain_sequences=chain_sequences, total_mass_da=total_mass)

built = build_pet_slab(N_CHAINS, CHAIN_REPEAT_UNITS)
universe = built.primitive
print(universe.hierarchy_summary(to_depth=2))
print(f"Mass: {built.total_mass_da:.1f} Da")
print(f"Slab density from chosen chains: {built.total_mass_da * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3:.3f} g/cm^3")

## 3. Finite-Geometry AA-DPD Slab Initialization

This initializer treats every atom as a DPD particle, labels one representative PET chain with OpenFF, maps bonds/angles/proper torsions/improper torsions into HOOMD, and relaxes the dense target 8 x 8 x 2 nm periodic cell. It follows the ionomer initializer pattern but omits counterions and temporary ion restraints.

In [ ]:
def random_rotation(rng: np.random.Generator) -> Rotation:
    return Rotation.random(random_state=rng)

def random_unit_vector(rng: np.random.Generator) -> np.ndarray:
    vector = rng.normal(size=3)
    norm = np.linalg.norm(vector)
    if norm < 1.0e-12:
        return np.array([1.0, 0.0, 0.0])
    return vector / norm

def wrap_position(position: np.ndarray, box_lengths_a: np.ndarray) -> np.ndarray:
    return ((np.asarray(position, dtype=float) + 0.5 * box_lengths_a) % box_lengths_a) - 0.5 * box_lengths_a

def rotation_between_vectors(source: np.ndarray, target: np.ndarray) -> Rotation:
    source_norm = np.linalg.norm(source)
    target_norm = np.linalg.norm(target)
    if source_norm < 1.0e-12 or target_norm < 1.0e-12:
        return Rotation.identity()
    source_unit = np.asarray(source, dtype=float) / source_norm
    target_unit = np.asarray(target, dtype=float) / target_norm
    return Rotation.align_vectors([target_unit], [source_unit])[0]

def mapped_atom(residue: Primitive, map_number: int) -> Primitive | None:
    for atom in residue.children:
        atom_map = atom.metadata.get("molAtomMapNumber")
        if atom_map is not None and int(atom_map) == map_number:
            return atom
    return None

def transform_residue_atoms(residue: Primitive, rotation: Rotation, source_anchor: np.ndarray, target_anchor: np.ndarray) -> None:
    points = []
    for atom in residue.children:
        old_position = np.asarray(atom.shape.centroid, dtype=float)
        new_position = target_anchor + rotation.apply(old_position - source_anchor)
        atom.shape = PointCloud(new_position)
        points.append(new_position)
    residue.shape = PointCloud(np.vstack(points))

def segment_atoms(segment: Primitive) -> list[Primitive]:
    return [atom for residue in segment.children for atom in residue.children]

def rdkit_mols_from_universe(root: Primitive) -> list[Chem.Mol]:
    resname_map = {
        residue.label: residue.metadata.get("residue_name", "PET")
        for residue in PreOrderIter(root)
        if residue.role == PrimitiveRole.RESIDUE
    }
    return list(primitive_to_rdkit_mols(root, resname_map=resname_map, default_atom_position=np.zeros(3)))

def atom_chain_id(atom: Chem.Atom) -> str:
    if atom.HasProp("chain_id"):
        return str(atom.GetProp("chain_id"))
    info = atom.GetPDBResidueInfo()
    return info.GetChainId().strip() if info is not None else "A"

def atom_residue_id(atom: Chem.Atom) -> str:
    if atom.HasProp("residue_id"):
        return str(atom.GetIntProp("residue_id"))
    info = atom.GetPDBResidueInfo()
    return str(info.GetResidueNumber()) if info is not None else "1"

def openmm_topology_from_rdkit_mols(rdkit_mols: list[Chem.Mol], box_vectors_nm: np.ndarray | None = None):
    from openmm import unit as omm_unit
    from openmm.app import Topology, element

    topology = Topology()
    chain_cache = {}
    residue_cache = {}
    for mol in rdkit_mols:
        atom_lookup = {}
        for atom_idx, atom in enumerate(mol.GetAtoms()):
            chain_id = atom_chain_id(atom)
            chain = chain_cache.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chain_cache[chain_id] = chain
            residue_name = atom.GetProp("residue_name") if atom.HasProp("residue_name") else "PET"
            residue_id = atom_residue_id(atom)
            residue_key = (chain_id, residue_id, residue_name)
            residue = residue_cache.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_id)
                residue_cache[residue_key] = residue
            symbol = atom.GetSymbol()
            atom_lookup[atom_idx] = topology.addAtom(f"{symbol}{atom_idx + 1}", element.get_by_symbol(symbol), residue)
        for bond in mol.GetBonds():
            topology.addBond(atom_lookup[bond.GetBeginAtomIdx()], atom_lookup[bond.GetEndAtomIdx()])
    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    return topology

def whole_packed_positions_a(positions_a: np.ndarray, rdkit_mols: list[Chem.Mol], box_lengths_a: np.ndarray, centered_input: bool = False) -> np.ndarray:
    packed_positions = np.zeros_like(positions_a, dtype=float)
    offset = 0
    for mol in rdkit_mols:
        n_atoms = mol.GetNumAtoms()
        wrapped = np.asarray(positions_a[offset:offset + n_atoms], dtype=float)
        unwrapped = wrapped.copy()
        adjacency = [[] for _ in range(n_atoms)]
        for bond in mol.GetBonds():
            begin_idx = bond.GetBeginAtomIdx()
            end_idx = bond.GetEndAtomIdx()
            adjacency[begin_idx].append(end_idx)
            adjacency[end_idx].append(begin_idx)
        visited = np.zeros(n_atoms, dtype=bool)
        for root_idx in range(n_atoms):
            if visited[root_idx]:
                continue
            visited[root_idx] = True
            stack = [root_idx]
            while stack:
                atom_idx = stack.pop()
                for neighbor_idx in adjacency[atom_idx]:
                    if visited[neighbor_idx]:
                        continue
                    delta = wrapped[neighbor_idx] - wrapped[atom_idx]
                    delta -= np.round(delta / box_lengths_a) * box_lengths_a
                    unwrapped[neighbor_idx] = unwrapped[atom_idx] + delta
                    visited[neighbor_idx] = True
                    stack.append(neighbor_idx)
        shifted = unwrapped + (0.5 * box_lengths_a if centered_input else 0.0)
        shifted -= np.floor(shifted.min(axis=0) / box_lengths_a) * box_lengths_a
        packed_positions[offset:offset + n_atoms] = shifted
        offset += n_atoms
    return packed_positions

def initialize_chain_positions(root: Primitive, rng: np.random.Generator) -> None:
    box_a = np.array([10.0 * SLAB_X_NM, 10.0 * SLAB_Y_NM, 10.0 * DPD_BOX_Z_NM], dtype=float)
    for segment in root.children:
        previous_tail_position = None
        walk_direction = random_unit_vector(rng)
        first_anchor = rng.uniform(-0.5 * box_a, 0.5 * box_a)
        for residue_idx, residue in enumerate(segment.children):
            head_atom = mapped_atom(residue, 1)
            tail_atom = mapped_atom(residue, 2)
            if residue_idx == 0:
                anchor_atom = tail_atom if tail_atom is not None else residue.children[0]
                source_anchor = np.asarray(anchor_atom.shape.centroid, dtype=float)
                rotation = random_rotation(rng)
                transform_residue_atoms(residue, rotation, source_anchor, first_anchor)
            else:
                if head_atom is None:
                    raise ValueError(f"Residue {residue.label!r} is missing connector atom map 1")
                source_head = np.asarray(head_atom.shape.centroid, dtype=float)
                if tail_atom is not None:
                    current_axis = np.asarray(tail_atom.shape.centroid, dtype=float) - source_head
                    walk_direction = random_unit_vector(rng)
                else:
                    current_axis = walk_direction
                target_head = previous_tail_position + INITIAL_REPEAT_SPACING_A * walk_direction
                rotation = rotation_between_vectors(current_axis, walk_direction)
                transform_residue_atoms(residue, rotation, source_head, target_head)
            if tail_atom is not None:
                previous_tail_position = np.asarray(tail_atom.shape.centroid, dtype=float)
            else:
                previous_tail_position = np.asarray(head_atom.shape.centroid, dtype=float)
        for atom in segment_atoms(segment):
            atom.shape = PointCloud(wrap_position(atom.shape.centroid, box_a))
        for residue in segment.children:
            residue.shape = PointCloud(np.vstack([np.asarray(atom.shape.centroid, dtype=float) for atom in residue.children]))

def close_contact_count(positions: np.ndarray, minimum_distance_a: float) -> int:
    if len(positions) < 2:
        return 0
    distances, _ = cKDTree(positions).query(positions, k=2)
    return int(np.count_nonzero(distances[:, 1] < minimum_distance_a))

def openff_labels_for_representative_chain(rdkit_mol: Chem.Mol):
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.units import unit as off_unit

    off_mol = Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
    labels = ForceField(FORCE_FIELD).label_molecules(Topology.from_molecules([off_mol]))[0]
    return labels, off_unit

def periodic_terms(parameter, off_unit, scale: float, phase_name: str) -> list[dict]:
    terms = []
    for term_idx, k_term in enumerate(parameter.k):
        idivf_values = getattr(parameter, "idivf", None)
        idivf = float(idivf_values[term_idx]) if idivf_values is not None else 1.0
        k_value = float(k_term.m_as(off_unit.kilocalorie_per_mole)) * scale / idivf
        terms.append({
            "k": abs(k_value),
            "d": 1 if k_value >= 0 else -1,
            "n": int(parameter.periodicity[term_idx]),
            phase_name: float(parameter.phase[term_idx].m_as(off_unit.radian)),
        })
    return terms

def openff_dpd_tables(labels, off_unit) -> dict:
    atom_types_by_local = {}
    epsilon_by_type = {}
    for (local_idx,), parameter in labels["vdW"].items():
        atom_type = f"T{local_idx}_{parameter.id}"
        atom_types_by_local[local_idx] = atom_type
        epsilon_by_type[atom_type] = float(parameter.epsilon.m_as(off_unit.kilocalorie_per_mole))
    epsilon_reference = max(epsilon_by_type.values())
    particle_types = sorted(epsilon_by_type)
    pair_params = {}
    for i, type_i in enumerate(particle_types):
        for type_j in particle_types[i:]:
            epsilon_ij = math.sqrt(epsilon_by_type[type_i] * epsilon_by_type[type_j])
            scale = epsilon_ij / epsilon_reference if epsilon_reference > 0 else 1.0
            pair_params[(type_i, type_j)] = {"A": DPD_A * scale, "gamma": DPD_GAMMA * scale}

    bond_params = {}
    bond_type_by_local = {}
    for local_pair, parameter in labels["Bonds"].items():
        i, j = tuple(local_pair)
        bond_type = f"b_{i}_{j}_{parameter.id}"
        bond_type_by_local[tuple(sorted((i, j)))] = bond_type
        bond_params[bond_type] = {
            "k": float(parameter.k.m_as(off_unit.kilocalorie_per_mole / off_unit.angstrom**2)) * DPD_BOND_K_SCALE,
            "r0": float(parameter.length.m_as(off_unit.angstrom)),
        }

    angle_params = {}
    angle_type_by_local = {}
    for local_triplet, parameter in labels["Angles"].items():
        i, j, k = tuple(local_triplet)
        angle_type = f"a_{i}_{j}_{k}_{parameter.id}"
        angle_type_by_local[(i, j, k)] = angle_type
        angle_params[angle_type] = {
            "k": float(parameter.k.m_as(off_unit.kilocalorie_per_mole / off_unit.radian**2)) * DPD_ANGLE_K_SCALE,
            "t0": float(parameter.angle.m_as(off_unit.radian)),
        }

    dihedral_params = {}
    dihedral_terms_by_local = {}
    for local_quad, parameter in labels.get("ProperTorsions", {}).items():
        quad = tuple(local_quad)
        terms = []
        for term_idx, params in enumerate(periodic_terms(parameter, off_unit, DPD_DIHEDRAL_K_SCALE, "phi0")):
            dihedral_type = f"d_{quad[0]}_{quad[1]}_{quad[2]}_{quad[3]}_{parameter.id}_{term_idx}"
            dihedral_params[dihedral_type] = params
            terms.append(dihedral_type)
        dihedral_terms_by_local[quad] = terms

    improper_params = {}
    improper_terms_by_local = {}
    for local_quad, parameter in labels.get("ImproperTorsions", {}).items():
        quad = tuple(local_quad)
        terms = []
        for term_idx, params in enumerate(periodic_terms(parameter, off_unit, DPD_IMPROPER_K_SCALE, "chi0")):
            improper_type = f"i_{quad[0]}_{quad[1]}_{quad[2]}_{quad[3]}_{parameter.id}_{term_idx}"
            improper_params[improper_type] = params
            terms.append(improper_type)
        improper_terms_by_local[quad] = terms

    return {
        "atom_types_by_local": atom_types_by_local,
        "particle_types": particle_types,
        "pair_params": pair_params,
        "bond_type_by_local": bond_type_by_local,
        "bond_params": bond_params,
        "angle_type_by_local": angle_type_by_local,
        "angle_params": angle_params,
        "dihedral_terms_by_local": dihedral_terms_by_local,
        "dihedral_params": dihedral_params,
        "improper_terms_by_local": improper_terms_by_local,
        "improper_params": improper_params,
    }

def run_finite_geometry_aa_dpd(root: Primitive) -> list[float]:
    if not HOOMD_AVAILABLE:
        raise RuntimeError(f"HOOMD is required for AA-DPD placement: {HOOMD_IMPORT_ERROR}")
    rng = np.random.default_rng(RANDOM_SEED)
    initialize_chain_positions(root, rng)
    rdkit_mols = rdkit_mols_from_universe(root)
    labels, off_unit = openff_labels_for_representative_chain(rdkit_mols[0])
    tables = openff_dpd_tables(labels, off_unit)
    atoms = []
    bonds = []
    bond_typeids = []
    angles = []
    angle_typeids = []
    dihedrals = []
    dihedral_typeids = []
    impropers = []
    improper_typeids = []
    bond_types = sorted(tables["bond_params"])
    angle_types = sorted(tables["angle_params"])
    dihedral_types = sorted(tables["dihedral_params"])
    improper_types = sorted(tables["improper_params"])
    bond_type_to_id = {name: idx for idx, name in enumerate(bond_types)}
    angle_type_to_id = {name: idx for idx, name in enumerate(angle_types)}
    dihedral_type_to_id = {name: idx for idx, name in enumerate(dihedral_types)}
    improper_type_to_id = {name: idx for idx, name in enumerate(improper_types)}
    offset = 0
    for segment, mol in zip(root.children, rdkit_mols):
        seg_atoms = segment_atoms(segment)
        if mol.GetNumAtoms() != len(seg_atoms):
            raise ValueError(f"Atom-order mismatch for {segment.label}: RDKit={mol.GetNumAtoms()}, MuPT={len(seg_atoms)}")
        atoms.extend(seg_atoms)
        for local_pair, bond_type in tables["bond_type_by_local"].items():
            bonds.append(tuple(offset + idx for idx in local_pair))
            bond_typeids.append(bond_type_to_id[bond_type])
        for local_triplet, angle_type in tables["angle_type_by_local"].items():
            angles.append(tuple(offset + idx for idx in local_triplet))
            angle_typeids.append(angle_type_to_id[angle_type])
        for local_quad, dihedral_terms in tables["dihedral_terms_by_local"].items():
            for dihedral_type in dihedral_terms:
                dihedrals.append(tuple(offset + idx for idx in local_quad))
                dihedral_typeids.append(dihedral_type_to_id[dihedral_type])
        for local_quad, improper_terms in tables["improper_terms_by_local"].items():
            for improper_type in improper_terms:
                impropers.append(tuple(offset + idx for idx in local_quad))
                improper_typeids.append(improper_type_to_id[improper_type])
        offset += len(seg_atoms)
    positions = np.vstack([np.asarray(atom.shape.centroid, dtype=float) for atom in atoms])
    particle_types = tables["particle_types"]
    type_to_id = {particle_type: idx for idx, particle_type in enumerate(particle_types)}
    typeid = np.array([type_to_id[tables["atom_types_by_local"][idx % rdkit_mols[0].GetNumAtoms()]] for idx in range(len(atoms))], dtype=np.uint32)

    frame = gsd.hoomd.Frame()
    frame.particles.N = len(atoms)
    frame.particles.types = particle_types
    frame.particles.typeid = typeid
    frame.particles.position = positions
    frame.particles.mass = np.array([float(atom.element.mass) for atom in atoms], dtype=float)
    frame.bonds.N = len(bonds)
    frame.bonds.types = bond_types
    frame.bonds.typeid = np.asarray(bond_typeids, dtype=np.uint32)
    frame.bonds.group = np.asarray(bonds, dtype=np.uint32)
    frame.angles.N = len(angles)
    frame.angles.types = angle_types
    frame.angles.typeid = np.asarray(angle_typeids, dtype=np.uint32)
    frame.angles.group = np.asarray(angles, dtype=np.uint32)
    frame.dihedrals.N = len(dihedrals)
    frame.dihedrals.types = dihedral_types
    frame.dihedrals.typeid = np.asarray(dihedral_typeids, dtype=np.uint32)
    frame.dihedrals.group = np.asarray(dihedrals, dtype=np.uint32)
    frame.impropers.N = len(impropers)
    frame.impropers.types = improper_types
    frame.impropers.typeid = np.asarray(improper_typeids, dtype=np.uint32)
    frame.impropers.group = np.asarray(impropers, dtype=np.uint32)
    box_parameters = [10.0 * SLAB_X_NM, 10.0 * SLAB_Y_NM, 10.0 * DPD_BOX_Z_NM, 0.0, 0.0, 0.0]
    frame.configuration.box = box_parameters
    box_lengths_a = np.asarray(box_parameters[:3], dtype=float)
    box_vectors_nm = np.diag([SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM])
    dcd_topology = openmm_topology_from_rdkit_mols(rdkit_mols, box_vectors_nm=box_vectors_nm)
    dpd_dcd_path = OUTPUT_ROOT / "pet_slab_aa_dpd_whole_packed.dcd"
    dpd_dcd_path.parent.mkdir(parents=True, exist_ok=True)
    dpd_dcd_handle = dpd_dcd_path.open("wb")
    from openmm import unit as omm_unit
    from openmm.app import DCDFile
    dpd_dcd = DCDFile(dpd_dcd_handle, dcd_topology, DPD_DT * omm_unit.picosecond, interval=max(1, DPD_STEPS_MAX // TRAJECTORY_FRAMES))

    integrator = hoomd.md.Integrator(dt=DPD_DT)
    integrator.methods.append(hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All()))
    harmonic = hoomd.md.bond.Harmonic()
    for bond_type, params in tables["bond_params"].items():
        harmonic.params[bond_type] = params
    angle_force = hoomd.md.angle.Harmonic()
    for angle_type, params in tables["angle_params"].items():
        angle_force.params[angle_type] = params
    dihedral_force = hoomd.md.dihedral.Periodic()
    for dihedral_type, params in tables["dihedral_params"].items():
        dihedral_force.params[dihedral_type] = params
    improper_force = hoomd.md.improper.Periodic()
    for improper_type, params in tables["improper_params"].items():
        improper_force.params[improper_type] = params
    nlist = hoomd.md.nlist.Cell(buffer=0.4, exclusions=("bond", "angle", "dihedral"))
    dpd = hoomd.md.pair.DPD(nlist=nlist, default_r_cut=DPD_R_CUT_A, kT=DPD_KT)
    for pair, params in tables["pair_params"].items():
        dpd.params[pair] = params
    integrator.forces.extend([harmonic, angle_force, dihedral_force, improper_force, dpd])
    print(
        f"HOOMD force table sizes: bonds={len(bonds)}, angles={len(angles)}, "
        f"dihedrals={len(dihedrals)}, impropers={len(impropers)}, particle_types={len(particle_types)}"
    )

    simulation = hoomd.Simulation(device=hoomd.device.CPU(), seed=int(rng.integers(1, 65000)))
    simulation.operations.integrator = integrator
    simulation.create_state_from_snapshot(frame)
    simulation.run(0)
    total_steps = 0
    start = time.perf_counter()
    try:
        for frame_idx in range(TRAJECTORY_FRAMES):
            target_step = int(round((frame_idx + 1) * DPD_STEPS_MAX / TRAJECTORY_FRAMES))
            steps = max(1, target_step - total_steps)
            simulation.run(steps)
            total_steps += steps
            snapshot = simulation.state.get_snapshot()
            positions = np.asarray(snapshot.particles.position, dtype=float)
            packed_nm = whole_packed_positions_a(positions, rdkit_mols, box_lengths_a, centered_input=True) * 0.1
            dpd_dcd.writeModel(packed_nm * omm_unit.nanometer, periodicBoxVectors=box_vectors_nm * omm_unit.nanometer)
            n_close = close_contact_count(positions, MIN_INTERMOLECULAR_DISTANCE_A)
            if (frame_idx + 1) % 10 == 0 or frame_idx == TRAJECTORY_FRAMES - 1:
                print(f"AA-DPD steps={total_steps}, close contacts < {MIN_INTERMOLECULAR_DISTANCE_A:.2f} A: {n_close}")
    finally:
        dpd_dcd_handle.close()
    snapshot = simulation.state.get_snapshot()
    final_positions = np.asarray(snapshot.particles.position, dtype=float)
    for atom, position in zip(atoms, final_positions):
        atom.shape = PointCloud(position)
    for segment in root.children:
        for residue in segment.children:
            residue.shape = PointCloud(np.vstack([np.asarray(atom.shape.centroid, dtype=float) for atom in residue.children]))
    root.metadata["unit_cell_parameters"] = box_parameters
    root.metadata["aa_dpd_steps"] = str(total_steps)
    root.metadata["aa_dpd_elapsed_s"] = str(time.perf_counter() - start)
    root.metadata["aa_dpd_close_contacts"] = str(close_contact_count(final_positions, MIN_INTERMOLECULAR_DISTANCE_A))
    root.metadata["aa_dpd_dcd"] = str(dpd_dcd_path.relative_to(EXAMPLES_ROOT))
    return box_parameters

if RUN_AA_DPD:
    box_parameters = run_finite_geometry_aa_dpd(universe)
else:
    initialize_chain_positions(universe, np.random.default_rng(RANDOM_SEED))
    box_parameters = [10.0 * SLAB_X_NM, 10.0 * SLAB_Y_NM, 10.0 * DPD_BOX_Z_NM, 0.0, 0.0, 0.0]
    universe.metadata["unit_cell_parameters"] = box_parameters
print(f"DPD/OpenMM box parameters in A: {box_parameters}")

## 4. Export SDF and Pre-Minimization CIF

In [ ]:
def make_rdkit_molecule_whole(mol: Chem.Mol, box_parameters_a: list[float] | tuple[float, ...] | None, centered_input: bool = True) -> None:
    """Unwrap each bonded molecule across an orthorhombic periodic box in-place."""
    if box_parameters_a is None or mol.GetNumAtoms() <= 1:
        return
    box_lengths = np.asarray(box_parameters_a[:3], dtype=float)
    if box_lengths.shape != (3,) or np.any(box_lengths <= 0) or not np.all(np.isfinite(box_lengths)):
        return
    conf = mol.GetConformer()
    wrapped_positions = np.asarray(conf.GetPositions(), dtype=float)
    unwrapped_positions = wrapped_positions.copy()
    adjacency = [[] for _ in range(mol.GetNumAtoms())]
    for bond in mol.GetBonds():
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()
        adjacency[begin_idx].append(end_idx)
        adjacency[end_idx].append(begin_idx)
    visited = np.zeros(mol.GetNumAtoms(), dtype=bool)
    for root_idx in range(mol.GetNumAtoms()):
        if visited[root_idx]:
            continue
        visited[root_idx] = True
        stack = [root_idx]
        while stack:
            atom_idx = stack.pop()
            for neighbor_idx in adjacency[atom_idx]:
                if visited[neighbor_idx]:
                    continue
                delta = wrapped_positions[neighbor_idx] - wrapped_positions[atom_idx]
                delta -= np.round(delta / box_lengths) * box_lengths
                unwrapped_positions[neighbor_idx] = unwrapped_positions[atom_idx] + delta
                visited[neighbor_idx] = True
                stack.append(neighbor_idx)
    # Keep each molecule whole, then choose the periodic image whose lower
    # coordinate corner lies in the displayed 0..L cell for PyMOL.
    unwrapped_positions = unwrapped_positions + (0.5 * box_lengths if centered_input else 0.0)
    unwrapped_positions -= np.floor(unwrapped_positions.min(axis=0) / box_lengths) * box_lengths
    for atom_idx, position in enumerate(unwrapped_positions):
        conf.SetAtomPosition(atom_idx, Point3D(float(position[0]), float(position[1]), float(position[2])))

def atom_chain_id(atom: Chem.Atom) -> str:
    if atom.HasProp("chain_id"):
        return str(atom.GetProp("chain_id"))
    info = atom.GetPDBResidueInfo()
    return info.GetChainId().strip() if info is not None else "A"

def atom_residue_id(atom: Chem.Atom) -> str:
    if atom.HasProp("residue_id"):
        return str(atom.GetIntProp("residue_id"))
    info = atom.GetPDBResidueInfo()
    return str(info.GetResidueNumber()) if info is not None else "1"

def write_rdkit_mols_to_pdbx(rdkit_mols: list[Chem.Mol], output_path: Path, box_vectors_nm: np.ndarray | None = None) -> None:
    import openmm
    from openmm import unit as omm_unit
    from openmm.app import PDBxFile, Topology, element
    topology = Topology()
    positions = []
    chain_cache = {}
    residue_cache = {}
    for mol_idx, mol in enumerate(rdkit_mols):
        atom_lookup = {}
        conf = mol.GetConformer()
        for atom_idx, atom in enumerate(mol.GetAtoms()):
            chain_id = atom_chain_id(atom)
            chain = chain_cache.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chain_cache[chain_id] = chain
            residue_name = atom.GetProp("residue_name") if atom.HasProp("residue_name") else "PET"
            residue_id = atom_residue_id(atom)
            residue_key = (chain_id, residue_id, residue_name)
            residue = residue_cache.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_id)
                residue_cache[residue_key] = residue
            symbol = atom.GetSymbol()
            atom_lookup[atom_idx] = topology.addAtom(f"{symbol}{atom_idx + 1}", element.get_by_symbol(symbol), residue)
            p = conf.GetAtomPosition(atom_idx)
            positions.append(openmm.Vec3(p.x * 0.1, p.y * 0.1, p.z * 0.1))
        for bond in mol.GetBonds():
            topology.addBond(atom_lookup[bond.GetBeginAtomIdx()], atom_lookup[bond.GetEndAtomIdx()])
    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    with output_path.open("w") as handle:
        PDBxFile.writeFile(topology, positions * omm_unit.nanometer, handle)

sdf_dir = OUTPUT_ROOT / "sdf"
sdf_dir.mkdir(parents=True, exist_ok=True)
rdkit_mols = rdkit_mols_from_universe(universe)
for mol in rdkit_mols:
    make_rdkit_molecule_whole(mol, box_parameters)
sdf_path = sdf_dir / "pet_slab_8x8x2nm_aa_dpd.sdf"
writer = Chem.SDWriter(str(sdf_path))
for mol in rdkit_mols:
    prepare_mupt_sdf_atom_props(mol)
    writer.write(mol)
writer.close()
box_vectors_nm = np.diag([SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM])
pre_min_cif_path = OUTPUT_ROOT / "pet_slab_8x8x2nm_aa_dpd_pre_min.cif"
write_rdkit_mols_to_pdbx(rdkit_mols, pre_min_cif_path, box_vectors_nm=box_vectors_nm)
manifest = {
    "target_density_g_cm3": TARGET_DENSITY_G_CM3,
    "actual_chain_density_g_cm3": built.total_mass_da * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3,
    "slab_dimensions_nm": [SLAB_X_NM, SLAB_Y_NM, SLAB_Z_NM],
    "dpd_openmm_box_nm": [SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM],
    "n_chains": N_CHAINS,
    "chain_repeat_units": CHAIN_REPEAT_UNITS,
    "aa_dpd_dcd": universe.metadata.get("aa_dpd_dcd"),
    "sdf": str(sdf_path.relative_to(EXAMPLES_ROOT)),
    "pre_min_cif": str(pre_min_cif_path.relative_to(EXAMPLES_ROOT)),
}
manifest_path = OUTPUT_ROOT / "pet_slab_8x8x2nm_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Wrote {len(rdkit_mols)} PET chain records to {sdf_path}")
print(f"Wrote pre-minimized CIF to {pre_min_cif_path}")
print(f"Wrote manifest to {manifest_path}")

## 5. OpenFF/OpenMM Minimization and Final CIF

In [ ]:
if RUN_OPENMM_MINIMIZATION:
    from openff.interchange import Interchange
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.toolkit.utils import ToolkitRegistry
    from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
    from openff.units import unit as off_unit
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat
    from openmm import unit as omm_unit
    from openmm.app import DCDFile, Simulation

    def transfer_metadata(rdkit_mol: Chem.Mol, off_mol: Molecule) -> None:
        for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
            props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
            off_atom.metadata.update({
                "residue_name": str(props.get("residue_name", "PET")),
                "residue_number": str(props.get("residue_id", props.get("mupt_residue_index", "1"))),
                "chain_id": str(props.get("chain_id", "A")),
                "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
            })

    def build_openff_topology_from_instances(rdkit_mols: list[Chem.Mol]):
        instance_molecules = []
        positions = []
        for rdkit_mol in rdkit_mols:
            off_mol = Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
            transfer_metadata(rdkit_mol, off_mol)
            instance_molecules.append(off_mol)
            positions.append(off_mol.conformers[0].m_as(off_unit.angstrom))
        template = instance_molecules[0]
        if NAGLToolkitWrapper.is_available():
            template.assign_partial_charges(
                partial_charge_method=PARTIAL_CHARGE_METHOD,
                toolkit_registry=ToolkitRegistry([NAGLToolkitWrapper()]),
            )
        else:
            raise RuntimeError("OpenFF NAGL is required for fast PET charge assignment in this notebook")
        topology = Topology.from_molecules([template] * len(instance_molecules))
        return topology, np.vstack(positions) * off_unit.angstrom, [template]

    topology, positions, charge_molecules = build_openff_topology_from_instances(rdkit_mols)
    force_field = ForceField(FORCE_FIELD)
    interchange = force_field.create_interchange(topology, charge_from_molecules=charge_molecules)
    interchange.positions = positions
    interchange.box = box_vectors_nm * off_unit.nanometer
    openmm_system = interchange.to_openmm(combine_nonbonded_forces=True)
    openmm_topology = interchange.to_openmm_topology()
    openmm_topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    integrator = LangevinMiddleIntegrator(OPENMM_MD_TEMPERATURE_K * omm_unit.kelvin, OPENMM_MD_FRICTION_PER_PS / omm_unit.picosecond, OPENMM_MD_TIMESTEP_FS * omm_unit.femtosecond)
    try:
        platform = openmm.Platform.getPlatformByName(OPENMM_PLATFORM_NAME)
        simulation = Simulation(openmm_topology, openmm_system, integrator, platform, OPENMM_PLATFORM_PROPERTIES)
    except Exception as exc:
        print(f"OpenMM {OPENMM_PLATFORM_NAME} unavailable; using default platform ({exc})")
        simulation = Simulation(openmm_topology, openmm_system, integrator)
    simulation.context.setPositions(interchange.positions.to_openmm())
    state0 = simulation.context.getState(getEnergy=True)
    print(f"Initial potential energy: {state0.getPotentialEnergy()}")
    simulation.minimizeEnergy(tolerance=OPENMM_MINIMIZATION_TOLERANCE_KJ_MOL_NM * omm_unit.kilojoule_per_mole / omm_unit.nanometer)
    state = simulation.context.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=False)
    minimized_energy_kj_mol = state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
    print(f"Minimized potential energy: {state.getPotentialEnergy()}")
    if not np.isfinite(minimized_energy_kj_mol):
        raise RuntimeError(f"OpenMM minimization produced non-finite potential energy: {state.getPotentialEnergy()}")
    minimized_positions = state.getPositions(asNumpy=True)
    simulation.context.setPositions(minimized_positions)
    def write_packed_dcd_frame(dcd: DCDFile, state) -> float:
        positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
        box_vectors_nm_current = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
        box_lengths_a = np.array([np.linalg.norm(vector) for vector in box_vectors_nm_current], dtype=float) * 10.0
        packed_nm = whole_packed_positions_a(positions_nm * 10.0, rdkit_mols, box_lengths_a, centered_input=False) * 0.1
        dcd.writeModel(packed_nm * omm_unit.nanometer, periodicBoxVectors=box_vectors_nm_current * omm_unit.nanometer)
        return state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)

    def run_md_segment(label: str, total_steps: int, dcd_path: Path) -> tuple[object, float]:
        print(f"Running {label}: steps={total_steps}, target_frames={TRAJECTORY_FRAMES}")
        start = time.perf_counter()
        steps_done = 0
        last_state = None
        with dcd_path.open("wb") as handle:
            dcd = DCDFile(handle, openmm_topology, OPENMM_MD_TIMESTEP_FS * omm_unit.femtosecond, interval=max(1, total_steps // TRAJECTORY_FRAMES))
            for frame_idx in range(TRAJECTORY_FRAMES):
                target_step = int(round((frame_idx + 1) * total_steps / TRAJECTORY_FRAMES))
                steps = max(1, target_step - steps_done)
                simulation.step(steps)
                steps_done += steps
                state_i = simulation.context.getState(getEnergy=True, getPositions=True, getVelocities=True, enforcePeriodicBox=False)
                potential_kj_mol = state_i.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
                positions_nm = state_i.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
                velocities_nm_ps = state_i.getVelocities(asNumpy=True).value_in_unit(omm_unit.nanometer / omm_unit.picosecond)
                if not (np.isfinite(potential_kj_mol) and np.all(np.isfinite(positions_nm)) and np.all(np.isfinite(velocities_nm_ps))):
                    raise RuntimeError(f"OpenMM {label} became unstable at step {steps_done}: potential={potential_kj_mol} kJ/mol")
                potential_kj_mol = write_packed_dcd_frame(dcd, state_i)
                last_state = state_i
                if (frame_idx + 1) % 10 == 0 or frame_idx == TRAJECTORY_FRAMES - 1:
                    print(f"{label} step {steps_done}/{total_steps}: potential={potential_kj_mol:.3f} kJ/mol")
        print(f"Completed {label} in {(time.perf_counter() - start) / 60.0:.2f} min; wrote {dcd_path}")
        return last_state, potential_kj_mol

    nvt_dcd_path = OUTPUT_ROOT / "pet_slab_openmm_nvt_whole_packed.dcd"
    npt_dcd_path = OUTPUT_ROOT / "pet_slab_openmm_npt_whole_packed.dcd"
    final_state, nvt_energy_kj_mol = run_md_segment("NVT", OPENMM_MD_STEPS, nvt_dcd_path)
    openmm_system.addForce(MonteCarloBarostat(OPENMM_NPT_PRESSURE_ATM * omm_unit.atmosphere, OPENMM_MD_TEMPERATURE_K * omm_unit.kelvin, OPENMM_BAROSTAT_FREQUENCY_STEPS))
    simulation.context.reinitialize(preserveState=True)
    final_state, npt_energy_kj_mol = run_md_segment("NPT", OPENMM_NPT_STEPS, npt_dcd_path)
    final_state = simulation.context.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=False)
    final_energy_kj_mol = final_state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
    if not np.isfinite(final_energy_kj_mol):
        raise RuntimeError(f"OpenMM final MD frame has non-finite potential energy: {final_state.getPotentialEnergy()}")
    print(f"Final MD potential energy: {final_state.getPotentialEnergy()}")
    final_box_vectors_nm = final_state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
    final_box_lengths_a = np.array([np.linalg.norm(vector) for vector in final_box_vectors_nm], dtype=float) * 10.0
    final_box_parameters = [float(final_box_lengths_a[0]), float(final_box_lengths_a[1]), float(final_box_lengths_a[2]), 0.0, 0.0, 0.0]
    final_positions_nm = final_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    for mol in rdkit_mols:
        conf = mol.GetConformer()
        for atom_idx in range(mol.GetNumAtoms()):
            p = final_positions_nm[atom_idx]
            conf.SetAtomPosition(atom_idx, Point3D(float(10.0 * p[0]), float(10.0 * p[1]), float(10.0 * p[2])))
        final_positions_nm = final_positions_nm[mol.GetNumAtoms():]
    for mol in rdkit_mols:
        make_rdkit_molecule_whole(mol, final_box_parameters, centered_input=False)
    final_cif_path = OUTPUT_ROOT / "output.cif"
    write_rdkit_mols_to_pdbx(rdkit_mols, final_cif_path, box_vectors_nm=final_box_vectors_nm)
    manifest["openmm_md_final_cif"] = str(final_cif_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_minimized_potential_energy"] = str(state.getPotentialEnergy())
    manifest["openmm_nvt_dcd"] = str(nvt_dcd_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_npt_dcd"] = str(npt_dcd_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_nvt_duration_ns"] = OPENMM_MD_DURATION_NS
    manifest["openmm_npt_duration_ns"] = OPENMM_NPT_DURATION_NS
    manifest["openmm_md_timestep_fs"] = OPENMM_MD_TIMESTEP_FS
    manifest["openmm_nvt_final_potential_energy_kj_mol"] = nvt_energy_kj_mol
    manifest["openmm_npt_final_potential_energy"] = str(final_state.getPotentialEnergy())
    manifest["openmm_npt_final_box_nm"] = final_box_vectors_nm.tolist()
    manifest["openmm_md_stable"] = True
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"Wrote stable-MD final CIF to {final_cif_path}")
else:
    print("OpenMM minimization skipped. Set RUN_OPENMM_MINIMIZATION = True to generate the final minimized CIF.")